# NTPC PPE Detection PoC: Stage 2 PPE Detector Training

This notebook trains a YOLOv11n object detector on the cropped person dataset prepared in the previous notebook. The model will run locally on your RTX 4070 GPU.

In [ ]:
# 1. Import dependencies and check GPU status
import torch
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("VRAM:", torch.cuda.get_device_properties(0).total_memory / (1024**3), "GB")

In [ ]:
# 2. Initialize YOLOv11 model
# We start with pretrained yolov11n (nano) or yolov11s (small)
# yolov11n is faster (ideal for target FPS), yolov11s has slightly better representation capacity.
model = YOLO('../yolov11n.pt') # downloads pretrained weights automatically

In [ ]:
# 3. Configure and start fine-tuning
# Optimized training params for safety apparel on RTX 4070 laptop GPU:
results = model.train(
    data='../datasets/ppe_crops/data.yaml', 
    epochs=30,             # Sufficient for fine-tuning on pre-trained weights
    imgsz=640,             # Standard resolution
    batch=16,              # Adjust depending on VRAM usage (16 is very safe for 8GB VRAM)
    device=0,              # Local GPU index 0
    workers=4,             # Multithreaded dataloading
    lr0=0.01,              # Initial learning rate
    optimizer='AdamW',     # Modern optimizer with better generalization
    save=True,             # Save checkpoints
    project='ntpc_ppe',    # Project folder
    name='stage2_crop_det',# Training run name
    val=True               # Validate after training epochs
)

In [ ]:
# 4. Validate Model & Print Metrics
metrics = model.val()
print("Validation Mean AP (mAP@50):", metrics.box.map50)
print("Validation Precision (per class):", metrics.box.p)
print("Validation Recall (per class):", metrics.box.r)

In [ ]:
# 5. Save and Export model weights
import os
import shutil

models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

best_weights = os.path.join(model.trainer.save_dir, 'best.pt')
target_weights = os.path.join(models_dir, 'ppe_crop_detector.pt')

if os.path.exists(best_weights):
    shutil.copy(best_weights, target_weights)
    print(f"Successfully saved fine-tuned model to: {target_weights}")
else:
    print("Error: best.pt weights not found. Check training runs path.")